## Simulating traffic passing multiple locks
In this notebook, we simulate two locks on a network which vessels from opposing direction shave to pass. Vessels are locked together if they can fit inside the lock, and arrive within the clustering time window.

#### 0. Import libraries

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core.utils import create_object
from opentnsim.graph import mixins as graph_module
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.output import HasOutput
from scipy.stats import norm, uniform, expon

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable
from opentnsim.environment.mixins.hydrodynamics import HydrodynamicData

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


In [2]:
%load_ext autoreload
%autoreload 2

#### 0. Create environment

In [3]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph

In [4]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.DiGraph()

# add nodes
graph.add_node('-2',geometry=transform(wgs84eqd_to_wgs84rad,Point(-350600,0)))
graph.add_node('-1',geometry=transform(wgs84eqd_to_wgs84rad,Point( -15000,0)))
graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
graph.add_node('+1',geometry=transform(wgs84eqd_to_wgs84rad,Point(15000,0)))
graph.add_node('+2',geometry=transform(wgs84eqd_to_wgs84rad,Point(350600,0)))

# add edges
graph.add_edge('-2','-1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-350600, 0),Point(-15000, 0)])), weight=1, length_m=350600)
graph.add_edge('-1','-2', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-15000, 0),Point(-350600, 0)])), weight=1, length_m=350600)

graph.add_edge('-1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-15000, 0),Point(-5000, 0)])), weight=1, length_m=10000)
graph.add_edge('0','-1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(-15000, 0)])), weight=1, length_m=10000)

graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(-5000, 0)])), weight=1, length_m=10000)

graph.add_edge('1','+1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(15000, 0)])), weight=1, length_m=10000)
graph.add_edge('+1','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(15000, 0),Point(5000, 0)])), weight=1, length_m=10000)

graph.add_edge('+2','+1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(15000, 0),Point(350600, 0)])), weight=1, length_m=350600)
graph.add_edge('+1','+2', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(350600, 0),Point(15000, 0)])), weight=1, length_m=350600)

# add graph to environment
env.graph = graph

In [5]:
graph_module.plot_graph(graph)

#### 1+ Adding infrastructure

In [6]:
lock_chamber_I = IsLockChamber(env=env,
                               lock_depth = 15,
                               name='Lock chamber I',
                               gate_open = '-1',
                               edge = ('-1','0'),
                               geometry_m = Polygon([Point(-10200, -25),Point(-10200, 25),Point(-9800, 25),Point(-9800, -25)]))

lock_chamber_II = IsLockChamber(env=env,
                                lock_depth = 15,
                                name='Lock chamber II',
                                gate_open = '-1',
                                edge = ('1','+1'),
                                geometry_m = Polygon([Point(9800, -25),Point(9800, 25),Point(10200, 25),Point(10200, -25)]))

In [7]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_area_A_I = IsLockWaitingArea(env=env,
                                     name = 'Waiting area A I',
                                     edge = ('-1','0'),
                                     distance_from_edge_start = 0)

waiting_area_B_I = IsLockWaitingArea(env=env,
                                     name = 'Waiting area B I',
                                     edge = ('0','-1'),
                                     distance_from_edge_start = 0)

waiting_area_A_II = IsLockWaitingArea(env=env,
                                      name = 'Waiting area A II',
                                      edge = ('1','+1'),
                                      distance_from_edge_start = 0)

waiting_area_B_II = IsLockWaitingArea(env=env,
                                      name = 'Waiting area B II',
                                      edge = ('+1','1'),
                                      distance_from_edge_start = 0)

In [8]:
lock_complex_I = IsLockComplex(lock_chambers = [lock_chamber_I],
                               waiting_areas = [waiting_area_A_I, waiting_area_B_I],
                               registration_nodes = ['-1','0'],
                               env=env,
                               name = 'Lock complex I',)

lock_complex_II = IsLockComplex(lock_chambers = [lock_chamber_II],
                                waiting_areas = [waiting_area_A_II, waiting_area_B_II],
                                registration_nodes = ['1','+1'],
                                env=env,
                                name = 'Lock complex II',)

#### 2. Create agents

In [9]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
        Identifiable,               # allows to give the object a name and a random ID,
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
    ), 
)

In [10]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [11]:
# create vessels from dict 
data_vessel_1 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 1",                                  # required by Identifiable
    "geometry": env.graph.nodes['-2']['geometry'],       # required by Locatable
    "route": nx.dijkstra_path(env.graph, "-2", "+2"),    # required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 10,                                             # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:00:00')  # required by PassesLockComplex
}  
vessel_1 = Vessel(**data_vessel_1)
vessel_1.name = 'Vessel 1'

data_vessel_2 = {
    "env": env,                                          # needed for simpy simulation
    "name": "Vessel 2",                                  # required by Identifiable
    "geometry": env.graph.nodes['+2']['geometry'],       # required by Locatable
    "route": nx.dijkstra_path(env.graph, "+2", "-2"),    # required by Routeable
    "v": 4,                                              # required by Movable, 4 m/s to check if the distance is covered in the expected time
    "L": 100,                                            # required by VesselProperties, interacts with the lock capacity
    "B": 20,                                             # required by VesselProperties
    "T": 10,                                             # required by VesselProperties
    "type": 'tanker',                                    # required by VesselProperties
    "arrival_time": pd.Timestamp('2025-01-01 00:05:00')  # required by PassesLockComplex
}  
vessel_2 = Vessel(**data_vessel_2)
vessel_2.name = 'Vessel 2'

#start the simulation
vessels = [vessel_1,vessel_2]
env.process(mission(env, vessel_1));
env.process(mission(env, vessel_2));

#### 3. Run simulation

In [12]:
env.run()

#### 4. Inspect output

In [13]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_complex_I.lock_chambers['Lock chamber I'].logbook)

print("'{}' logbook data:".format(lock_complex_I.name))  
print('')

display(lock_df)

'Lock complex I' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock gate closing start,2025-01-02 00:46:30.172786,{},-1
1,Lock gate closing stop,2025-01-02 00:51:30.172786,{},-1
2,Lock levelling start,2025-01-02 00:51:30.172786,{},-1
3,Lock levelling stop,2025-01-02 01:01:30.172786,{},0
4,Lock gate opening start,2025-01-02 01:01:30.172786,{},0
5,Lock gate opening stop,2025-01-02 01:06:30.172786,{},0
6,Lock gate closing start,2025-01-02 02:49:38.941685,{},0
7,Lock gate closing stop,2025-01-02 02:54:38.941685,{},0
8,Lock levelling start,2025-01-02 02:54:38.941685,{},0
9,Lock levelling stop,2025-01-02 03:04:38.941685,{},-1


In [14]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_complex_II.lock_chambers['Lock chamber II'].logbook)

print("'{}' logbook data:".format(lock_complex_II.name))  
print('')

display(lock_df)

'Lock complex II' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock gate closing start,2025-01-02 00:25:50.000000,{},1
1,Lock gate closing stop,2025-01-02 00:30:50.000000,{},1
2,Lock levelling start,2025-01-02 00:30:50.000000,{},1
3,Lock levelling stop,2025-01-02 00:40:50.000000,{},+1
4,Lock gate opening start,2025-01-02 00:40:50.000000,{},+1
5,Lock gate opening stop,2025-01-02 00:45:50.000000,{},+1
6,Lock gate closing start,2025-01-02 01:01:30.172786,{},+1
7,Lock gate closing stop,2025-01-02 01:06:30.172786,{},+1
8,Lock levelling start,2025-01-02 01:06:30.172786,{},+1
9,Lock levelling stop,2025-01-02 01:16:30.172786,{},1


#### Gantt chart of event table

In [15]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([*vessels, 
                                                            lock_complex_I.lock_chambers['Lock chamber I'], 
                                                            lock_complex_II.lock_chambers['Lock chamber II']])
fig = generate_vessel_gantt_chart(df_eventtable)

#### Time-distance diagram of vessels passing the lock and planning info

In [28]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_complex_I.lock_chambers['Lock chamber I'].plot(xlimmin = -5050, 
                                                          xlimmax = 5050,
                                                          ylimmin = pd.Timestamp('2025-01-02 00:00:00'),
                                                          ylimmax = pd.Timestamp('2025-01-02 04:00:00'),
                                                          method='Plotly',
                                                          boundary_nodes = ['-1','+1'])

fig.update_layout(height=cm_to_pixels(20))

In [27]:
# We can plot the time-distance diagram
fig = lock_complex_II.lock_chambers['Lock chamber II'].plot(xlimmin = -5050, 
                                                            xlimmax = 5050,
                                                            ylimmin = pd.Timestamp('2025-01-02 00:00:00'),
                                                            ylimmax = pd.Timestamp('2025-01-02 04:00:00'),
                                                            method='Plotly',
                                                            boundary_nodes = ['0','+1'])

fig.update_layout(height=cm_to_pixels(20))

#### Vessel delays: individual delays and overall average

In [18]:
delays = []
for vessel in vessels:
    vessel_df = pd.DataFrame(vessel.logbook)
    waiting_stop = vessel_df[vessel_df.Message == "Waiting stop"]
    if not waiting_stop.empty:
        delay = (waiting_stop.Timestamp-vessel.metadata["arrival_time"]).iloc[0]
    else:
        delay = pd.Timedelta(seconds=0)
    delays.append(delay)

In [19]:
print(f"The average vessel delay is {np.round(np.average(delays).total_seconds()/60,1)} minutes")

The average vessel delay is 0.0 minutes


In [20]:
pd.DataFrame(vessel.logbook)

,Message,Timestamp,Value,Geometry
0,Sailing from node +2 to node +1 start,2025-01-01 00:05:00.000000,0,POINT (3.149493386123042 0)
1,Sailing from node +2 to node +1 stop,2025-01-02 00:25:50.000000,350600,POINT (0.1347472926179282 0)
2,Sailing from node +1 to node 1 start,2025-01-02 00:25:50.000000,350600,POINT (0.1347472926179282 0)
3,Waiting for lock operation start,2025-01-02 00:25:50.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.1347472926179282 0)
4,Waiting for lock operation stop,2025-01-02 00:35:50.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.1347472926179282 0)
5,Sailing to first lock gate start,2025-01-02 00:35:50.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.1347472926179282 0)
6,Sailing to first lock gate stop,2025-01-02 00:55:50.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0916281589801912 0)
7,Sailing to position in lock start,2025-01-02 00:55:50.000000,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0916281589801912 0)
8,Sailing to position in lock stop,2025-01-02 01:01:30.172786,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0884840554857729 0)
9,Waiting for lock gate closing start,2025-01-02 01:01:30.172786,"{'origin': '', 'destination': '', 'route': [],...",POINT (0.0884840554857729 0)
